<a href="https://colab.research.google.com/github/vardhan23v/agentic-ai/blob/main/Hands_On_1_Data_Cleaning_and_Summary.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Hands-on 1: Clean and Summarize Sales Data

**Topic:** Python data handling with CSV and Pandas  
**Level:** Beginner  
**Time:** 45–50 minutes

## Scenario

An online shop has a sales CSV file. The file contains a duplicate order,
one missing price, inconsistent category names and one invalid quantity.

## What you will learn

- Load a CSV file.
- Check missing and duplicate values.
- Clean simple data problems.
- Filter and update rows.
- Create a category summary.
- Save the cleaned result as a new CSV.

## Step 1: Create the sample CSV dataset

This cell creates a real CSV file. Students can also open the provided
sales_data.csv file separately to view the raw dataset.

In [2]:
from pathlib import Path

# This cell creates the CSV file so the notebook can run directly in Colab.
Path("sales_data.csv").write_text(
    'order_id,order_date,product_id,product_name,category,price,quantity,payment_status\nO101,2026-08-01,1,Mascara, beauty ,799,2,Paid\nO102,2026-08-02,2,Eye Shadow,BEAUTY,1599,1,Paid\nO103,2026-08-03,3,Face Powder,Beauty,1199,3,Paid\nO104,2026-08-04,4,Lipstick,beauty,999,2,Pending\nO104,2026-08-04,4,Lipstick,beauty,999,2,Pending\nO105,2026-08-05,5,Nail Polish,BEAUTY,699,4,Paid\nO106,2026-08-06,6,CK One Perfume,fragrance,3999,1,Paid\nO107,2026-08-07,7,Coco Noir,Fragrances,,1,Paid\nO108,2026-08-08,8,Dior Jadore,FRAGRANCES,7499,1,Paid\nO109,2026-08-09,9,Dolce Shine,fragrance,5999,-1,Paid\nO110,2026-08-10,10,Gucci Bloom,Fragrances,6499,2,Failed\nO111,2026-08-11,1,Mascara,Beauty,849,2,Paid\n',
    encoding="utf-8"
)

print("Created: sales_data.csv")

Created: sales_data.csv


## Step 2: Load the CSV file

Pandas reads the CSV and stores it as a DataFrame. A DataFrame is a table
with rows and columns.

In [3]:
import pandas as pd

sales = pd.read_csv("sales_data.csv")

print("Number of rows and columns:", sales.shape)
print(sales.head().to_string(index=False))

Number of rows and columns: (12, 8)
order_id order_date  product_id product_name category  price  quantity payment_status
    O101 2026-08-01           1      Mascara  beauty   799.0         2           Paid
    O102 2026-08-02           2   Eye Shadow   BEAUTY 1599.0         1           Paid
    O103 2026-08-03           3  Face Powder   Beauty 1199.0         3           Paid
    O104 2026-08-04           4     Lipstick   beauty  999.0         2        Pending
    O104 2026-08-04           4     Lipstick   beauty  999.0         2        Pending


## Step 3: Inspect the dataset

We check missing values and duplicate order IDs before changing the data.

In [10]:
print("Missing values:")
print(sales.isnull().sum().to_string())

print("\nDuplicate order IDs:")
print(sales["order_id"].duplicated().sum())

Missing values:
order_id          0
order_date        0
product_id        0
product_name      0
category          0
price             1
quantity          0
payment_status    0

Duplicate order IDs:
1


## Step 4: Clean the data

Cleaning rules:

1. Remove duplicate order IDs.
2. Remove extra spaces and standardize category names.
3. Fill the missing price with the median price.
4. Keep only quantities greater than zero.

In [5]:
clean_sales = sales.copy()

clean_sales = clean_sales.drop_duplicates(subset="order_id")

clean_sales["category"] = (
    clean_sales["category"]
    .str.strip()
    .str.title()
    .replace({"Fragrance": "Fragrances"})
)

median_price = clean_sales["price"].median()
clean_sales["price"] = clean_sales["price"].fillna(median_price)

clean_sales = clean_sales[clean_sales["quantity"] > 0].copy()

print(clean_sales.to_string(index=False))

order_id order_date  product_id   product_name   category  price  quantity payment_status
    O101 2026-08-01           1        Mascara     Beauty  799.0         2           Paid
    O102 2026-08-02           2     Eye Shadow     Beauty 1599.0         1           Paid
    O103 2026-08-03           3    Face Powder     Beauty 1199.0         3           Paid
    O104 2026-08-04           4       Lipstick     Beauty  999.0         2        Pending
    O105 2026-08-05           5    Nail Polish     Beauty  699.0         4           Paid
    O106 2026-08-06           6 CK One Perfume Fragrances 3999.0         1           Paid
    O107 2026-08-07           7      Coco Noir Fragrances 1399.0         1           Paid
    O108 2026-08-08           8    Dior Jadore Fragrances 7499.0         1           Paid
    O110 2026-08-10          10    Gucci Bloom Fragrances 6499.0         2         Failed
    O111 2026-08-11           1        Mascara     Beauty  849.0         2           Paid


## Step 5: Calculate revenue

Revenue is price multiplied by quantity.

In [6]:
clean_sales["revenue"] = (
    clean_sales["price"] * clean_sales["quantity"]
)

print(
    clean_sales[
        ["order_id", "product_name", "price", "quantity", "revenue"]
    ].to_string(index=False)
)

order_id   product_name  price  quantity  revenue
    O101        Mascara  799.0         2   1598.0
    O102     Eye Shadow 1599.0         1   1599.0
    O103    Face Powder 1199.0         3   3597.0
    O104       Lipstick  999.0         2   1998.0
    O105    Nail Polish  699.0         4   2796.0
    O106 CK One Perfume 3999.0         1   3999.0
    O107      Coco Noir 1399.0         1   1399.0
    O108    Dior Jadore 7499.0         1   7499.0
    O110    Gucci Bloom 6499.0         2  12998.0
    O111        Mascara  849.0         2   1698.0


## Step 6: Filter and update rows

Find orders with revenue of at least ₹3,000. Then label orders with
revenue of at least ₹5,000 as High Value.

In [7]:
high_revenue_orders = clean_sales[
    clean_sales["revenue"] >= 3000
]

clean_sales["order_type"] = "Normal"
clean_sales.loc[
    clean_sales["revenue"] >= 5000,
    "order_type"
] = "High Value"

print("Orders with revenue of at least ₹3,000:")
print(
    high_revenue_orders[
        ["order_id", "product_name", "revenue"]
    ].to_string(index=False)
)

Orders with revenue of at least ₹3,000:
order_id   product_name  revenue
    O103    Face Powder   3597.0
    O106 CK One Perfume   3999.0
    O108    Dior Jadore   7499.0
    O110    Gucci Bloom  12998.0


## Step 7: Create a category summary

In [8]:
category_summary = (
    clean_sales.groupby("category")
    .agg(
        total_orders=("order_id", "count"),
        total_quantity=("quantity", "sum"),
        total_revenue=("revenue", "sum")
    )
    .reset_index()
)

print(category_summary.to_string(index=False))

  category  total_orders  total_quantity  total_revenue
    Beauty             6              14        13286.0
Fragrances             4               5        25895.0


## Step 8: Save the results

In [9]:
clean_sales.to_csv("cleaned_sales.csv", index=False)
category_summary.to_csv("category_summary.csv", index=False)

print("Created: cleaned_sales.csv")
print("Created: category_summary.csv")

Created: cleaned_sales.csv
Created: category_summary.csv


## Student task

1. Display only Paid orders.
2. Find the product with the highest revenue.
3. Change the High Value condition from ₹5,000 to ₹4,000.

## Expected learning

You can now load, inspect, clean, filter, update, summarize and save CSV data.